<a href="https://colab.research.google.com/github/gunvarawork-cpu/code-repository/blob/main/Lab_1__Pan_sharpening_6606614623.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pan-sharpening**

เทคนิค Pan-sharpening เป็นเทคนิคที่ใช้กับภาพถ่ายดาวเทียม ที่มี Panchromatic band เช่น THEOS-1, LANDSAT-8, LANDSAT-9 เพื่อให้มีรายละเอียดและมีประโยชน์มากขึ้น โดย Lab นี้เราจะทดลองกับข้อมูลดาวเทียม Landsat-9

ข้อมูลถ่ายที่เป็น Multi-Spectral มักจะมีรายละเอียดภาพไม่ค่อยดีนัก เมื่อเทียบกับข้อมูล Panchromatics  ดังนั้น การปรับความคมชัดของภาพช่วยได้โดยการรวมภาพเหล่านี้เข้ากับภาพขาวดำพิเศษที่มีรายละเอียดมากขึ้น ภาพขาวดำที่เรียกว่าภาพแพนโครมาติก มีรายละเอียดในระดับที่สูงกว่า แต่ไม่มีสี เราต้องการใช้รายละเอียดจากภาพแพนโครมาติกและสีจากภาพ Landsat-9 เพื่อสร้างภาพสีที่มีรายละเอียดสูงใหม่

ในการดำเนินการใน Lab นี้ นักศึกษา จำเป็นต้องตรวจสอบให้แน่ใจว่าข้อมูลทั้ง Panchromatic และ Multispectral ของ Landsat-9 อยู่ในตำแหน่งที่สมบูรณ์แบบ เพื่อให้เข้ากันได้อย่างลงตัว

มีหลายวิธีในการรวมภาพเหล่านี้ เช่น Brovey Transform, IHS Transform และ PCA วิธีการเหล่านี้ทำให้แน่ใจว่าภาพใหม่จะคงสีจาก Landsat-9 แต่ยังได้รับรายละเอียดเพิ่มเติมจากภาพแพนโครมาติกด้วย


เมื่อเราทำ Pan-sharpening เสร็จแล้ว เราก็จะได้ภาพใหม่ที่เป็นภาพสีที่มีรายละเอียดสูงขึ้น  โดยมีสีทั้งหมดจาก Landsat-9 และความคมชัดพิเศษจากภาพแบบแพนโครมาติก รูปภาพใหม่นี้สามารถนำไปใช้ได้หลายอย่าง เช่น ศึกษาพื้นดิน ค้นหาการเปลี่ยนแปลง หรือเพียงแค่ดูรายละเอียดเพิ่มเติมเกี่ยวกับโลก ดังนั้น การปรับความคมชัดของภาพเป็นวิธีหนึ่งในการทำให้ภาพจากดาวเทียมมีประโยชน์มากขึ้นสำหรับงานสำคัญทุกประเภท

In [1]:
# ทำการ Authenticate และ initialize Earth Engine
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='ee-gunvarawoon') #อย่าลืมเปลี่ยนชื่อโปรเจคของตัวเอง

In [2]:
# กำหนดพื้นที่สนใจ
geometry = ee.Geometry.Point([98.95799098999555, 18.84423947416328])

# เรียกภาพ L9 ตัวอย่างแล้วดึง RGB และ Pan ออกมา
image = (ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA")
         .filterDate('2022-01-01', '2022-03-30')
         .filterBounds(geometry)
         .sort('CLOUD_COVER')
         .first())

In [3]:
# ทำการ Pan-Sharp
rgb = image.select('B4', 'B3', 'B2')
pan = image.select('B8')

# แปลงเป็น HSV, สลับในแถบ PAN และแปลงกลับเป็น RGB
huesat = rgb.rgbToHsv().select('hue', 'saturation')
upres = ee.Image.cat([huesat, pan]).hsvToRgb()

In [4]:
# สร้างแผนที่
Map = geemap.Map(center=[18.84423947416328, 98.95799098999555], zoom=14)

# แสดง เลเยอร์ ก่อนและหลังโดยใช้พารามิเตอร์ vis เดียวกัน
Map.addLayer(rgb, {'max': 0.28}, 'Original')
Map.addLayer(pan, {'max': 0.28}, 'Pan')
Map.addLayer(upres, {'max': 0.28}, 'Pansharpened')
Map


Map(center=[18.84423947416328, 98.95799098999555], controls=(WidgetControl(options=['position', 'transparent_b…

คำถาม
ข้อเพื่อทดสอบความเข้าใจของนักศึกษาเกี่ยวกับเทคนิคการทำ Pan-sharpening ด้วยภาพ Landsat-9 จงตอบคำถามต่อไปนี้

1. อะไรคือเป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกล โดยเฉพาะเมื่อใช้ภาพ Landsat-9 อธิบายว่าเหตุใดจึงมีความสำคัญในการประมวลผลภาพ

2. อธิบายขั้นตอนสำคัญที่เกี่ยวข้องกับการปรับความคมชัดด้วนเทคนิค Pan-sharpening ตั้งแต่การรับข้อมูลไปจนถึงการสร้างภาพที่ปรับความคมชัด เทคนิค Pan-sharpening มี กระบวนการสุ่มตัวอย่างใหม่ช่วยจัดแนวภาพหลายสเปกตรัมและภาพแพนโครมาติกอย่างไร


**ตอบคำถาม**

**1. ตอบ**

  Pan-sharpening **มีเป้าหมายหลักเพื่อเพิ่มความละเอียดเชิงพื้นที่ของภาพ **Multispectral โดยอาศัยรายละเอียดจาก Panchromatic band ในกรณีใช้ Landsat-9 ภาพ Multispectral มีความละเอียด 30 เมตร ขณะที่ Panchromatic band มีความละเอียดสูงกว่า คือ 15 เมตร ซึ่งช่วยให้ได้ภาพสีที่มีความคมชัดมากขึ้น โดยยังคงข้อมูลเชิงสเปกตรัมของภาพ Multispectral ส่งผลให้การตีความและการวิเคราะห์ภาพถ่ายดาวเทียมมีความแม่นยำและมีประสิทธิภาพมากขึ้น

**2. ตอบ**
  อธิบายขั้นตอน Pan-sharpening ประกอบด้วย
*  **การรับข้อมูล**

    เลือกข้อมูลภาพจากดาวเทียม เช่น Landsat-9 ซึ่งต้องมีทั้ง:
    - ภาพ Multispectral (MS) : ให้ข้อมูลสี/สเปกตรัม
    - ภาพ Panchromatic (PAN) : ให้รายละเอียดเชิงพื้นที่สูง
*  **การคัดเลือกภาพ**

    เลือกภาพที่เหมาะสม เช่น
    - เมฆน้อย
    - วันเวลาตรงที่ต้องการศึกษา
*  **การเลือกแบนด์**
    - เลือก MS bands สำหรับแสดงสี (เช่น B4 B3 B2 → RGB)
    - เลือก PAN band (B8)
*  **การจัดแนวภาพ**

    ตรวจสอบให้แน่ใจว่า:
      - ภาพ MS และ PAN ซ้อนทับกันถูกต้อง
      - ไม่มีการเลื่อนตำแหน่ง (spatial shift)
*  **การ Resampling**

    เนื่องจาก:

    MS = 30 เมตร

    PAN = 15 เมตร

    ต้องปรับ pixel size ให้เท่ากันก่อนการรวมภาพ
*  **การผสานภาพ** (Image Fusion / Pan-sharpening)

    ใช้เทคนิคต่าง ๆ เช่น HSV / IHS / Brovey / PCA

    ตัวอย่าง HSV method:
      - แปลง RGB → HSV
      - แทนที่ Value ด้วย PAN
      - แปลงกลับ HSV → RGB
*  **การสร้างภาพผลลัพธ์** (Output Generation) ได้ภาพใหม่ที่:

    - เป็นภาพสี
    - ความละเอียดสูงขึ้น
    - รายละเอียดคมชัดกว่าเดิม
    
ซึ่ง**กระบวนการ Resampling หรือการสุ่มตัวอย่าง** ใน Pan-sharpening
มีความสำคัญมาก เพราะช่วยจัดภาพได้โดยการ:
- ปรับขนาดพิกเซลของ MS ให้ตรงกับ PAN
- จัด grid ของภาพให้สอดคล้องกัน
- ลดความคลาดเคลื่อนเชิงพื้นที่
- ทำให้การผสานภาพถูกต้อง ภาพไม่เบี้ยว ขอบไม่แตก สีไม่เพี้ยน